# MEXCLP: expected coverage when units are busy

A demand area can be geographically covered but still have no available unit. This lab uses a deliberately small exact solver to explore the effect of unit busy probability.

The model assumes equal busy fractions and independent unit availability. These are simplifying assumptions, not facts about every emergency-service system.

In [ ]:
from pathlib import Path
import sys
relative_package = Path('implementations/python/mexclp')
repo_root = next((candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / relative_package).is_dir()), None)
if repo_root is None:
    raise FileNotFoundError(f'Cannot locate {relative_package} from {Path.cwd()}')
sys.path.insert(0, str(repo_root / relative_package))
import matplotlib.pyplot as plt
from mexclp import solve_mexclp


## Define a small service area

Rows are demand areas; columns are candidate bases. Costs and the threshold must use the same unit. Demand weights represent the relative amount of service demand.

In [ ]:
costs = [[2, 8, 9], [7, 3, 8], [4, 6, 2], [8, 4, 3]]
demand = [100, 80, 60, 40]
threshold = 5
units = 3
busy_fraction = 0.25
sites, expected, contributions = solve_mexclp(costs, units, threshold, busy_fraction, demand_weights=demand)
print('Unit placements (candidate indices):', sites)
print('Expected covered demand:', round(expected, 2))
for i, value in enumerate(contributions):
    print(f'Demand area {i}: expected contribution {value:.2f}')


## How does the busy fraction change the result?

A unit that is busy more often contributes less expected availability. The solver can also change where units are placed as q changes.

In [ ]:
fractions = [0.0, 0.2, 0.4, 0.6, 0.8]
values = []
placements = []
for q in fractions:
    chosen, score, _ = solve_mexclp(costs, units, threshold, q, demand_weights=demand)
    placements.append(chosen)
    values.append(score)
    print(f'q={q:.1f}: expected coverage={score:.2f}; placements={chosen}')
plt.figure(figsize=(7, 4))
plt.plot(fractions, values, marker='o')
plt.xlabel('Common unit busy probability q')
plt.ylabel('Expected covered demand')
plt.title('Sensitivity to unit unavailability')
plt.grid(alpha=0.25)
plt.show()


## Interpret the result

Compare this result with deterministic maximal covering and r-covering. MEXCLP discounts demand according to the number of eligible units and the assumed busy fraction. It does not guarantee service, model queues, or account for correlated availability. A shared incident or common handover delay can make unit states dependent.

In [ ]:
q = 0.3
for k in range(4):
    probability = 1 - q**k
    print(f'{k} eligible units: {probability:.3f} probability at least one is available')
